# Generative Models — Implementations

The VAE and the GAN again with autograd doing the calculus, plus the canonical `torch.nn` GAN recipe. Every random draw — init weights, the VAE's eps, the GAN's z and batch indices — comes from one NumPy generator reseeded as the notebook's global `rng`, so every lane sees the same numbers in the same order and the equivalence deltas measure arithmetic, not luck.

## 19_vae

Encode to a distribution, sample it differentiably, decode the sample. *No library lane:* neither sklearn nor `torch.nn` ships a VAE estimator — the model is these five `Linear` layers plus a loss, and the torch lane already gives the honest comparison.

### torch

The same five-layer VAE on tensors: weights and the reparameterisation noise come from the shared NumPy `rng`, in the NumPy lane's exact draw order. **What torch adds:** one `total_loss.backward()` replaces five hand-derived gradient blocks — including the chain rule through `z = mu + eps*std`, which is the whole reason the reparameterisation trick exists.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Draw every random thing with NumPy — weights and eps — and as_tensor the result.
# 2. Keep the draw order: enc_l1, enc_mu, enc_logvar, dec_l1, dec_out, then eps per step.
# 3. The implied objective is 0.5*sum((recon-x)^2)/n + 0.5*sum(mu^2+e^lv-lv-1)/n.
# 4. One total loss, one backward(): autograd owns the chain rule through z = mu+eps*std.
# 5. float64 end to end; update() steps every layer and clears its grads.


class Linear:
    """The setup's Linear on tensors: same rng draw for W, autograd instead of dW."""

    def __init__(self, in_dim, out_dim):
        self.W = torch.as_tensor(rng.normal(0, 0.1, (in_dim, out_dim))).requires_grad_(True)
        self.b = torch.zeros(out_dim, dtype=torch.float64, requires_grad=True)

    def forward(self, x):
        return x @ self.W + self.b

    def update(self, lr):
        with torch.no_grad():
            self.W -= lr * self.W.grad
            self.b -= lr * self.b.grad
        self.W.grad = None
        self.b.grad = None


class VAE:
    """The notebook's VAE with the five hand-derived gradient blocks replaced by
    one backward(). Same layer names, same draw order, same objective."""

    def __init__(self, input_dim=2, hidden_dim=8, latent_dim=2):
        self.enc_l1 = Linear(input_dim, hidden_dim)
        self.enc_mu = Linear(hidden_dim, latent_dim)
        self.enc_logvar = Linear(hidden_dim, latent_dim)
        self.dec_l1 = Linear(latent_dim, hidden_dim)
        self.dec_out = Linear(hidden_dim, input_dim)

    def reparameterize(self, mu, logvar):
        """z = mu + eps*std with eps drawn from the shared NumPy rng — autograd
        differentiates through mu and std while eps stays a constant."""
        std = torch.exp(0.5 * logvar)
        eps = torch.as_tensor(rng.normal(0, 1, size=tuple(std.shape)))
        return mu + eps * std, eps, std

    def forward(self, x):
        h = torch.relu(self.enc_l1.forward(x))
        mu = self.enc_mu.forward(h)
        logvar = self.enc_logvar.forward(h)
        z, eps, std = self.reparameterize(mu, logvar)
        h_dec = torch.relu(self.dec_l1.forward(z))
        recon = self.dec_out.forward(h_dec)
        return recon, mu, logvar, z, eps, std

    def loss(self, x, recon, mu, logvar):
        """The objective the NumPy backward implies: per-sample squared error
        (times 1/2) plus the Gaussian KL, both averaged over the batch."""
        n = x.shape[0]
        recon_loss = 0.5 * torch.sum((recon - x) ** 2) / n
        kl_loss = 0.5 * torch.sum(mu ** 2 + torch.exp(logvar) - logvar - 1) / n
        return recon_loss + kl_loss, recon_loss, kl_loss

    def backward(self, x, recon, mu, logvar, z=None, eps=None, std=None):
        """Same call shape as the NumPy lane; z/eps/std are only kept for the
        mirror — autograd already knows the reparameterisation path."""
        total, recon_loss, kl_loss = self.loss(x, recon, mu, logvar)
        total.backward()
        return float(total.detach()), float(recon_loss.detach()), float(kl_loss.detach())

    def update(self, lr):
        for layer in [self.enc_l1, self.enc_mu, self.enc_logvar, self.dec_l1, self.dec_out]:
            layer.update(lr)


In [ ]:
# exports: final_loss, loss_tail, recon_head, mu_head
rng = np.random.default_rng(1907)
_means_eq = np.array([[2.0, 2.0], [-2.0, -2.0]])
_comp_eq = rng.integers(0, 2, size=64)
_X_eq = rng.normal(loc=_means_eq[_comp_eq], scale=0.5)
_Xt_eq = torch.as_tensor(_X_eq)

_vae_eq = VAE(input_dim=2, hidden_dim=8, latent_dim=2)
_hist_eq = []
for _step_eq in range(250):
    _recon_eq, _mu_eq, _lv_eq, _z_eq, _eps_eq, _std_eq = _vae_eq.forward(_Xt_eq)
    _tot_eq, _rl_eq, _kl_eq = _vae_eq.backward(_Xt_eq, _recon_eq, _mu_eq, _lv_eq)
    _hist_eq.append(_tot_eq)
    _vae_eq.update(0.05)

_recon_eq, _mu_eq, _lv_eq, _z_eq, _eps_eq, _std_eq = _vae_eq.forward(_Xt_eq)
_tot_fin, _rl_fin, _kl_fin = _vae_eq.loss(_Xt_eq, _recon_eq, _mu_eq, _lv_eq)
final_loss = float(_tot_fin.detach())
loss_tail = _hist_eq[-5:]
recon_head = _recon_eq[:5].detach().numpy()
mu_head = _mu_eq[:5].detach().numpy()
print(f"loss: {_hist_eq[0]:.4f} -> {final_loss:.4f}  (KL term {float(_kl_fin):.4f})")


In [ ]:
# Autograd reproduces the notebook's hand-derived KL gradients on leaf tensors.
_mu_t = torch.tensor([[0.3, -0.7], [0.1, 0.4]], dtype=torch.float64, requires_grad=True)
_lv_t = torch.tensor([[0.2, -0.1], [0.0, 0.5]], dtype=torch.float64, requires_grad=True)
_kl_t = 0.5 * torch.sum(_mu_t ** 2 + torch.exp(_lv_t) - _lv_t - 1) / _mu_t.shape[0]
_kl_t.backward()
assert torch.allclose(_mu_t.grad, _mu_t.detach() / _mu_t.shape[0], atol=1e-12), \
    "dKL/dmu must equal mu/n, the notebook's dmu_kl"
assert torch.allclose(_lv_t.grad, -0.5 * (1 - torch.exp(_lv_t.detach())) / _mu_t.shape[0],
                      atol=1e-12), "dKL/dlogvar must equal -0.5(1-e^logvar)/n"

# And the reparameterisation chain rule: for fixed eps, dz/dlogvar = 0.5*eps*std.
_lv2_t = torch.tensor([[0.3, -0.2]], dtype=torch.float64, requires_grad=True)
_eps2_t = torch.tensor([[0.7, -1.1]], dtype=torch.float64)
(1.5 + _eps2_t * torch.exp(0.5 * _lv2_t)).sum().backward()
assert torch.allclose(_lv2_t.grad, 0.5 * _eps2_t * torch.exp(0.5 * _lv2_t.detach()),
                      atol=1e-12), "dz/dlogvar must equal 0.5*eps*std, the notebook's factor"

assert final_loss < _hist_eq[0], "250 steps must reduce the ELBO-style objective"
assert float(_kl_fin) >= 0.0 and float(_rl_fin) >= 0.0, "both loss parts are non-negative"

# The sample is a deterministic function of the NumPy seed — the property the
# cross-lane equivalence relies on.
rng = np.random.default_rng(0)
_z1_eq = _vae_eq.reparameterize(_mu_eq, _lv_eq)[0]
rng = np.random.default_rng(0)
_z2_eq = _vae_eq.reparameterize(_mu_eq, _lv_eq)[0]
assert float(torch.max(torch.abs(_z1_eq - _z2_eq))) == 0.0, "same seed, same z"


## 19_gan

Two networks against each other; training is alternating gradient steps, so once the draws are shared the whole game is deterministic.

### torch

Same Generator and Discriminator on tensors, same alternating loop: real batch then fake batch for D (an update after each), then one non-saturating step for G through the updated D. **What torch adds:** autograd carries both losses back through the sigmoid and both networks — the scratch lane's cache discipline (whose forward last filled `self.x`?) disappears.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Probabilities, not logits: D ends in sigmoid and BCE clips at 1e-12, like the scratch.
# 2. Detach the fake batch in the D step — the scratch never pushes D's grads into G there.
# 3. The G step backprops through D on purpose; update only G, then drop D's stale grads.
# 4. Every z and every batch index comes from the shared NumPy rng, in the same order.


class Linear:
    """The setup's Linear on tensors: same rng draw for W, autograd instead of dW."""

    def __init__(self, in_dim, out_dim):
        self.W = torch.as_tensor(rng.normal(0, 0.1, (in_dim, out_dim))).requires_grad_(True)
        self.b = torch.zeros(out_dim, dtype=torch.float64, requires_grad=True)

    def forward(self, x):
        return x @ self.W + self.b

    def update(self, lr):
        with torch.no_grad():
            self.W -= lr * self.W.grad
            self.b -= lr * self.b.grad
        self.W.grad = None
        self.b.grad = None


class Generator:
    """z -> data space; same two Linear draws as the NumPy lane."""

    def __init__(self, latent_dim=2, hidden_dim=8, out_dim=2):
        self.l1 = Linear(latent_dim, hidden_dim)
        self.out = Linear(hidden_dim, out_dim)

    def forward(self, z):
        return self.out.forward(torch.relu(self.l1.forward(z)))

    def update(self, lr):
        self.l1.update(lr)
        self.out.update(lr)


class Discriminator:
    """data -> probability of real; sigmoid output, matching bce's convention."""

    def __init__(self, in_dim=2, hidden_dim=8):
        self.l1 = Linear(in_dim, hidden_dim)
        self.out = Linear(hidden_dim, 1)

    def forward(self, x):
        return torch.sigmoid(self.out.forward(torch.relu(self.l1.forward(x))))

    def update(self, lr):
        self.l1.update(lr)
        self.out.update(lr)


def bce_loss_and_grad(pred, target):
    """The scratch BCE on probabilities, clip included. The returned grad is the
    notebook's hand formula — training uses loss.backward() instead, and the
    checks confirm the two agree."""
    epsilon = 1e-12
    p = torch.clamp(pred, epsilon, 1 - epsilon)
    loss = -torch.mean(target * torch.log(p) + (1 - target) * torch.log(1 - p))
    with torch.no_grad():
        grad = (p - target) / (p * (1 - p) * p.shape[0])
    return loss, grad


In [ ]:
# exports: d_loss_tail, g_loss_tail, sample_head
rng = np.random.default_rng(1909)
_means_eq = np.array([[2.0, 2.0], [-2.0, -2.0]])
_comp_eq = rng.integers(0, 2, size=96)
_X_eq = rng.normal(loc=_means_eq[_comp_eq], scale=0.5)
_Xt_eq = torch.as_tensor(_X_eq)

def _zero_grads_eq(model):
    for _lyr in (model.l1, model.out):
        _lyr.W.grad = None
        _lyr.b.grad = None

_G_eq = Generator()
_D_eq = Discriminator()
_d_hist_eq, _g_hist_eq = [], []
for _epoch_eq in range(150):
    # --- D step: real batch, update, then fake batch, update ---
    _idx_eq = rng.integers(0, 96, size=32)
    _real_eq = _Xt_eq[_idx_eq]
    _zd_eq = torch.as_tensor(rng.normal(0, 1, (32, 2)))
    _fake_eq = _G_eq.forward(_zd_eq)
    _pred_real_eq = _D_eq.forward(_real_eq)
    _dlr_eq, _ = bce_loss_and_grad(_pred_real_eq, torch.ones_like(_pred_real_eq))
    _dlr_eq.backward()
    _D_eq.update(0.05)
    _pred_fake_eq = _D_eq.forward(_fake_eq.detach())
    _dlf_eq, _ = bce_loss_and_grad(_pred_fake_eq, torch.zeros_like(_pred_fake_eq))
    _dlf_eq.backward()
    _D_eq.update(0.05)
    # --- G step: fool the updated D ---
    _zg_eq = torch.as_tensor(rng.normal(0, 1, (32, 2)))
    _fake_eq = _G_eq.forward(_zg_eq)
    _pred_g_eq = _D_eq.forward(_fake_eq)
    _gl_eq, _ = bce_loss_and_grad(_pred_g_eq, torch.ones_like(_pred_g_eq))
    _gl_eq.backward()
    _G_eq.update(0.05)
    _zero_grads_eq(_D_eq)  # D took part in the G backward but must not keep it
    _d_hist_eq.append(float(_dlr_eq.detach()) + float(_dlf_eq.detach()))
    _g_hist_eq.append(float(_gl_eq.detach()))

_zs_eq = torch.as_tensor(rng.normal(0, 1, (32, 2)))
_sample_eq = _G_eq.forward(_zs_eq).detach()
d_loss_tail = _d_hist_eq[-5:]
g_loss_tail = _g_hist_eq[-5:]
sample_head = _sample_eq[:5].numpy()
print(f"D loss {_d_hist_eq[0]:.4f} -> {_d_hist_eq[-1]:.4f}, "
      f"G loss {_g_hist_eq[0]:.4f} -> {_g_hist_eq[-1]:.4f}")


In [ ]:
# Autograd agrees with the notebook's hand-derived BCE gradient.
_p_t = torch.tensor([[0.3], [0.8], [0.55]], dtype=torch.float64, requires_grad=True)
_t_t = torch.tensor([[1.0], [0.0], [1.0]], dtype=torch.float64)
_l_t, _g_hand = bce_loss_and_grad(_p_t, _t_t)
_l_t.backward()
assert torch.allclose(_p_t.grad, _g_hand, atol=1e-12), \
    "autograd's dBCE/dpred must match (p-t)/(p(1-p)n)"

with torch.no_grad():
    _p_all_eq = _D_eq.forward(_Xt_eq)
assert float(_p_all_eq.min()) > 0.0 and float(_p_all_eq.max()) < 1.0, \
    "a sigmoid discriminator outputs probabilities"

# After training, D still ranks real data above generated data on average.
with torch.no_grad():
    _zc_eq = torch.as_tensor(rng.normal(0, 1, (200, 2)))
    _p_fake_c = _D_eq.forward(_G_eq.forward(_zc_eq))
assert float(_p_all_eq.mean()) > float(_p_fake_c.mean()), \
    "D(real) should exceed D(fake) on average"

# The trained G moved its mass from near the origin toward the data modes.
_G0_eq = Generator()
with torch.no_grad():
    _z0_eq = torch.as_tensor(rng.normal(0, 1, (200, 2)))
    _norm_trained = float(torch.linalg.norm(_G_eq.forward(_z0_eq), dim=1).mean())
    _norm_fresh = float(torch.linalg.norm(_G0_eq.forward(_z0_eq), dim=1).mean())
assert _norm_trained > _norm_fresh, "training pushed samples away from the 0.1-scale init"


### library

The canonical PyTorch recipe: `nn.Sequential` networks, `nn.BCELoss`, `optim.SGD`, weights copied from the same NumPy draws — transposed, because `nn.Linear` stores `(out, in)`. **What the library adds:** plumbing only. `nn.BCELoss` is exactly the scratch loss on probabilities and SGD is exactly `p -= lr*grad`, so the deltas should read ~1e-15.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

# hints:
# 1. nn.Linear stores weight as (out, in): copy the NumPy draw transposed, zero the bias.
# 2. nn.BCELoss on probabilities is exactly the scratch bce_loss_and_grad's loss.
# 3. SGD without momentum is p -= lr*grad; set the group lr, step, zero_grad.
# 4. D.opt.zero_grad() after the G step replaces hand-zeroing D's stale grads.


class Generator:
    """The same generator as an nn.Sequential, seeded from the same NumPy draws."""

    def __init__(self, latent_dim=2, hidden_dim=8, out_dim=2):
        self.net = nn.Sequential(nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
                                 nn.Linear(hidden_dim, out_dim)).double()
        with torch.no_grad():
            self.net[0].weight.copy_(
                torch.as_tensor(rng.normal(0, 0.1, (latent_dim, hidden_dim)).T))
            self.net[0].bias.zero_()
            self.net[2].weight.copy_(
                torch.as_tensor(rng.normal(0, 0.1, (hidden_dim, out_dim)).T))
            self.net[2].bias.zero_()
        self.opt = torch.optim.SGD(self.net.parameters(), lr=0.05)

    def forward(self, z):
        return self.net(z)

    def update(self, lr):
        for group in self.opt.param_groups:
            group["lr"] = lr
        self.opt.step()
        self.opt.zero_grad()


class Discriminator:
    """nn.Sequential ending in nn.Sigmoid — probabilities out, as bce expects."""

    def __init__(self, in_dim=2, hidden_dim=8):
        self.net = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(),
                                 nn.Linear(hidden_dim, 1), nn.Sigmoid()).double()
        with torch.no_grad():
            self.net[0].weight.copy_(
                torch.as_tensor(rng.normal(0, 0.1, (in_dim, hidden_dim)).T))
            self.net[0].bias.zero_()
            self.net[2].weight.copy_(
                torch.as_tensor(rng.normal(0, 0.1, (hidden_dim, 1)).T))
            self.net[2].bias.zero_()
        self.opt = torch.optim.SGD(self.net.parameters(), lr=0.05)

    def forward(self, x):
        return self.net(x)

    def update(self, lr):
        for group in self.opt.param_groups:
            group["lr"] = lr
        self.opt.step()
        self.opt.zero_grad()


def bce_loss_and_grad(pred, target):
    """nn.BCELoss supplies the loss; the hand gradient rides along so the checks
    can hold the library to the notebook's formula."""
    loss = nn.BCELoss()(pred, target)
    with torch.no_grad():
        p = torch.clamp(pred, 1e-12, 1 - 1e-12)
        grad = (p - target) / (p * (1 - p) * p.shape[0])
    return loss, grad


In [ ]:
# exports: d_loss_tail, g_loss_tail, sample_head
rng = np.random.default_rng(1909)
_means_eq = np.array([[2.0, 2.0], [-2.0, -2.0]])
_comp_eq = rng.integers(0, 2, size=96)
_X_eq = rng.normal(loc=_means_eq[_comp_eq], scale=0.5)
_Xt_eq = torch.as_tensor(_X_eq)

_G_eq = Generator()
_D_eq = Discriminator()
_d_hist_eq, _g_hist_eq = [], []
for _epoch_eq in range(150):
    # --- D step: real batch, update, then fake batch, update ---
    _idx_eq = rng.integers(0, 96, size=32)
    _real_eq = _Xt_eq[_idx_eq]
    _zd_eq = torch.as_tensor(rng.normal(0, 1, (32, 2)))
    _fake_eq = _G_eq.forward(_zd_eq)
    _pred_real_eq = _D_eq.forward(_real_eq)
    _dlr_eq, _ = bce_loss_and_grad(_pred_real_eq, torch.ones_like(_pred_real_eq))
    _dlr_eq.backward()
    _D_eq.update(0.05)
    _pred_fake_eq = _D_eq.forward(_fake_eq.detach())
    _dlf_eq, _ = bce_loss_and_grad(_pred_fake_eq, torch.zeros_like(_pred_fake_eq))
    _dlf_eq.backward()
    _D_eq.update(0.05)
    # --- G step: fool the updated D ---
    _zg_eq = torch.as_tensor(rng.normal(0, 1, (32, 2)))
    _fake_eq = _G_eq.forward(_zg_eq)
    _pred_g_eq = _D_eq.forward(_fake_eq)
    _gl_eq, _ = bce_loss_and_grad(_pred_g_eq, torch.ones_like(_pred_g_eq))
    _gl_eq.backward()
    _G_eq.update(0.05)
    _D_eq.opt.zero_grad()  # D took part in the G backward but must not keep it
    _d_hist_eq.append(float(_dlr_eq.detach()) + float(_dlf_eq.detach()))
    _g_hist_eq.append(float(_gl_eq.detach()))

_zs_eq = torch.as_tensor(rng.normal(0, 1, (32, 2)))
_sample_eq = _G_eq.forward(_zs_eq).detach()
d_loss_tail = _d_hist_eq[-5:]
g_loss_tail = _g_hist_eq[-5:]
sample_head = _sample_eq[:5].numpy()
print(f"D loss {_d_hist_eq[0]:.4f} -> {_d_hist_eq[-1]:.4f}, "
      f"G loss {_g_hist_eq[0]:.4f} -> {_g_hist_eq[-1]:.4f}")


In [ ]:
# nn.BCELoss really is the scratch formula on probabilities.
_p_c = torch.tensor([[0.2], [0.9], [0.4]], dtype=torch.float64)
_t_c = torch.tensor([[0.0], [1.0], [1.0]], dtype=torch.float64)
_manual_c = -torch.mean(_t_c * torch.log(_p_c) + (1 - _t_c) * torch.log(1 - _p_c))
assert abs(float(nn.BCELoss()(_p_c, _t_c)) - float(_manual_c)) < 1e-12, \
    "nn.BCELoss must equal -mean(t log p + (1-t) log(1-p))"

assert tuple(_D_eq.net[0].weight.shape) == (8, 2), "nn.Linear keeps weight as (out, in)"
assert isinstance(_D_eq.net[3], nn.Sigmoid), "D ends in probabilities, matching bce"

# After training, D still ranks real data above generated data on average.
with torch.no_grad():
    _p_real_c = _D_eq.forward(_Xt_eq)
    _zc_eq = torch.as_tensor(rng.normal(0, 1, (200, 2)))
    _p_fake_c = _D_eq.forward(_G_eq.forward(_zc_eq))
assert float(_p_real_c.mean()) > float(_p_fake_c.mean()), \
    "D(real) should exceed D(fake) on average"
